In [2]:
import torch

In [3]:
# toy dataset

X_train = torch.tensor([
    [-1.2, 3.1],
    [-0.9, 2.9],
    [-0.5, 2.6],
    [2.3, -1.1],
    [2.7, -1.5]
])

y_train = torch.tensor([0, 0, 0, 1, 1])

In [4]:
X_test = torch.tensor([
    [-0.8, 2.8],
    [2.6, -1.6],
])

y_test = torch.tensor([0, 1])

Class labels in PyTorch is required to start from 0. That is, if we are havivng two classes, the label should be 0 and 1. Also, the number of the output nodes be equal to the number of classes, or the highest label value + 1.

In [5]:
from torch.utils.data import Dataset # PyTorch Dataset class


class ToyDataset(Dataset):
    def __init__(self, X, y):
        """
        Sets up the data attributes that can be accessed using the class methods __getitem__ and __len__
        """
        self.features = X
        self.labels = y

    def __getitem__(self, index):
        """
        Instruction to return exactly one item from the dataset using an index
        """
        one_x = self.features[index]
        one_y = self.labels[index]
        return one_x, one_y

    def __len__(self):
        """
        To get the length of the dataset
        """
        return self.labels.shape[0]

In [7]:
from torch.utils.data import DataLoader # to sample from the dataset

torch.manual_seed(123)

# This helps in defining the PyTorch dataset
train_ds = ToyDataset(X_train, y_train)

train_loader = DataLoader(
    dataset=train_ds,
    batch_size=2,
    shuffle=True,
    num_workers=0
)


test_ds = ToyDataset(X_test, y_test)

test_loader = DataLoader(
    dataset=test_ds,
    batch_size=2,
    shuffle=False,
    num_workers=0
)

In [8]:
# Iterate over the data loader just to show the tensor per batch
for idx, (x, y) in enumerate(train_loader):
    print(f"Batch {idx+1}:", x, y)

Batch 1: tensor([[ 2.3000, -1.1000],
        [-0.9000,  2.9000]]) tensor([1, 0])
Batch 2: tensor([[-1.2000,  3.1000],
        [-0.5000,  2.6000]]) tensor([0, 0])
Batch 3: tensor([[ 2.7000, -1.5000]]) tensor([1])


The iteration is repeated to demonstrate a change in shuffling when the dataset is iterated the second time, even with the `torch.manual_seed`.
Also note that because we have a single element in the last batch because we have batch size to be 5-element datasets

In [9]:
for idx, (x, y) in enumerate(train_loader):
    print(f"Batch {idx+1}:", x, y)

Batch 1: tensor([[-1.2000,  3.1000],
        [-0.5000,  2.6000]]) tensor([0, 0])
Batch 2: tensor([[ 2.3000, -1.1000],
        [-0.9000,  2.9000]]) tensor([1, 0])
Batch 3: tensor([[ 2.7000, -1.5000]]) tensor([1])


In [10]:
# To avoid disturbing convergence on the last batch of a training epoch, set `drop_last` to True

train_loader = DataLoader(
    dataset=train_ds,
    batch_size=2,
    shuffle=True,
    num_workers=0,
    drop_last=True
)

# let's check
for idx, (x, y) in enumerate(train_loader):
    print(f"Batch {idx+1}:", x, y)

Batch 1: tensor([[-0.9000,  2.9000],
        [ 2.3000, -1.1000]]) tensor([0, 1])
Batch 2: tensor([[ 2.7000, -1.5000],
        [-0.5000,  2.6000]]) tensor([1, 0])


The `num_workers` need to be set to value greater than 0 so that multiple worker processes are launched to load the data in parrallel. This frees the main process to focus on model training and better system resource utilization

A typical training loop

In [11]:
import torch

class NeuralNetwork(torch.nn.Module):
    def __init__(self, num_inputs, num_outputs):
        super().__init__()

        self.layers = torch.nn.Sequential(

            # 1st hidden layer
            torch.nn.Linear(num_inputs, 30),
            torch.nn.ReLU(),

            # 2nd hidden layer
            torch.nn.Linear(30, 20),
            torch.nn.ReLU(),

            # output layer
            torch.nn.Linear(20, num_outputs),
        )

    def forward(self, x):
        logits = self.layers(x)
        return logits

In [13]:
import torch.nn.functional as F


torch.manual_seed(123)
model = NeuralNetwork(num_inputs=2, num_outputs=2)
optimizer = torch.optim.SGD(model.parameters(), lr=0.5) 
# Stochastic gradient descent (SDG) as an optimizer, learning rate (hyperparameter) of 0.5, should ensure
# convergence after certain epochs

num_epochs = 3 # Also an hyperparameter

for epoch in range(num_epochs):

    model.train() # Put the model into training mode
    for batch_idx, (features, labels) in enumerate(train_loader):

        logits = model(features)
        
        # Logits directly to the cross_entropy  loss function which applies the softmax function internally
        loss = F.cross_entropy(logits, labels) # Loss function. 

        # Needs to be included for every iteration to reset the gradient to zero
        optimizer.zero_grad()
        
        # Calculates the computation graph gradient in the background
        loss.backward()

        # Uses the gradient to update the model parameters and minimize the loss
        # For SGD used, it means multiplying the gradients with the learning rate and adding the scaled
        # negative gradients to the parameters
        optimizer.step()

        ### LOGGING
        print(f"Epoch: {epoch+1:03d}/{num_epochs:03d}"
              f" | Batch {batch_idx:03d}/{len(train_loader):03d}"
              f" | Train/Val Loss: {loss:.2f}")

    model.eval() # Evaluation mode
    # Optional model evaluation

Epoch: 001/003 | Batch 000/002 | Train/Val Loss: 0.75
Epoch: 001/003 | Batch 001/002 | Train/Val Loss: 0.65
Epoch: 002/003 | Batch 000/002 | Train/Val Loss: 0.44
Epoch: 002/003 | Batch 001/002 | Train/Val Loss: 0.13
Epoch: 003/003 | Batch 000/002 | Train/Val Loss: 0.03
Epoch: 003/003 | Batch 001/002 | Train/Val Loss: 0.00


When the loss reaches 0, it means that the model converges on the training set

When the loss reaches 0, it means that the model converges on the training set. Asides from train and test sets, we may have the validation set to help in tweaking the hyperparameter by being used multiple times, but the test set once so as to avoid biasing the evaluation. The `model.train()` and `model.eval()` are to put the model into training and evaluation modes, respectively. This is important for components such as dropout and batch normalization layers that behave differently. It is also helpful to include them to avoid unexpected behavior when the model architecture is changed or code is reused to train a different model.

In [ ]:
model.eval()

with torch.no_grad():
    outputs = model(X_train)

print(outputs)

In [14]:
model.eval()

with torch.no_grad():
    outputs = model(X_train)

print(outputs)

tensor([[ 2.8569, -4.1618],
        [ 2.5382, -3.7548],
        [ 2.0944, -3.1820],
        [-1.4814,  1.4816],
        [-1.7176,  1.7342]])


In [16]:
# Class membership probabilities
torch.set_printoptions(sci_mode=False)
probas = torch.softmax(outputs, dim=1)
print(probas)

tensor([[    0.9991,     0.0009],
        [    0.9982,     0.0018],
        [    0.9949,     0.0051],
        [    0.0491,     0.9509],
        [    0.0307,     0.9693]])


Reading the output above: first row suggests that the training example has the 99.91% probability of belonging to class 0 and 0.09% probability of being class 1. The `set_printoptions` makes the outputs more legible.

In [17]:
# PyTorch argmax used to convert this into class labels. dim=1 gives the highest value in each row. 
# And dim=1 gives the highest value in each column

predictions = torch.argmax(probas, dim=1)
print(predictions) # Coming from the X_train

tensor([0, 0, 0, 1, 1])


In [18]:
# argmax can be directly applied to the logits (outputs) directly. Not neccesarily on the softmax probas
predictions = torch.argmax(outputs, dim=1)
print(predictions)

tensor([0, 0, 0, 1, 1])


In [19]:
# Checking the predicted labels with the actual labels
predictions == y_train

tensor([True, True, True, True, True])

In [20]:
# Checking the sum of correct predictions

torch.sum(predictions == y_train)

tensor(5)

In [24]:
# compute_accuracy function to generalize the prediction accuracy

def compute_accuracy(model, dataloader):
    """
    Computes the model's accuracy 
    Params
    -------
    model - the trained model
    dataloader - dataloader object of the dataset created from the PyTorch Dataloader class
    returns
    accuracy (int)
    -------
    """

    model = model.eval()
    correct = 0.0
    total_examples = 0

    for idx, (features, labels) in enumerate(dataloader):

        with torch.no_grad():
            logits = model(features)

        predictions = torch.argmax(logits, dim=1)
        compare = labels == predictions
        correct += torch.sum(compare)
        total_examples += len(compare)

    return (correct / total_examples).item()

In [22]:
# for the train_loader - for check
compute_accuracy(model, train_loader)

1.0

In [23]:
# the test loader, which is the ideal
compute_accuracy(model, test_loader)

1.0

Saving and loading models

In [26]:
# Saving model - the python's state_dict maps each layer in the model to its trainable parameters 
# (weights and biases)
torch.save(model.state_dict(), "model.pth") # .pt

In [27]:
#m Load model
model = NeuralNetwork(2, 2) # needs to match the original model exactly
model.load_state_dict(torch.load("model.pth", weights_only=True))

<All keys matched successfully>